In [55]:
import pandas as pd

In [56]:
df=pd.read_csv("../data/raw/station_delays_clean.csv")

In [57]:
df.head()

,journey_date,train_number,train_name,train_type,category,source_station,destination_station,station_sequence,station_code,station_name,distance_km,platform,status,scheduled_arrival,actual_arrival,scheduled_departure,actual_departure,arrival_delay_minutes,departure_delay_minutes
0,01-05-2026,12951,New Delhi Tejas Rajdhani Express,Rajdhani Express,Premium,MMCT,NDLS,1,MMCT,Mumbai Central,0.0,1.0,departed,NaN,NaN,2026-05-01T17:00:00+05:30,NaN,NaN,NaN
1,01-05-2026,12951,New Delhi Tejas Rajdhani Express,Rajdhani Express,Premium,MMCT,NDLS,19,BVI,Borivali,29.5,6.0,departed,2026-05-01T17:20:00+05:30,2026-05-01T17:32:07+05:30,2026-05-01T17:22:00+05:30,2026-05-01T17:38:08+05:30,13.0,16.0
2,01-05-2026,12951,New Delhi Tejas Rajdhani Express,Rajdhani Express,Premium,MMCT,NDLS,62,ST,Surat,263.0,1.0,departed,2026-05-01T19:43:00+05:30,2026-05-01T19:58:13+05:30,2026-05-01T19:48:00+05:30,2026-05-01T20:04:49+05:30,14.0,19.0
3,01-05-2026,12951,New Delhi Tejas Rajdhani Express,Rajdhani Express,Premium,MMCT,NDLS,88,BRC,Vadodara Junction,392.7,2.0,departed,2026-05-01T21:06:00+05:30,2026-05-01T21:25:10+05:30,2026-05-01T21:16:00+05:30,2026-05-01T21:37:13+05:30,17.0,20.0
4,01-05-2026,12951,New Delhi Tejas Rajdhani Express,Rajdhani Express,Premium,MMCT,NDLS,127,RTM,Ratlam Junction,652.8,5.0,departed,2026-05-02T00:25:00+05:30,2026-05-02T00:32:05+05:30,2026-05-02T00:28:00+05:30,2026-05-02T00:39:21+05:30,7.0,13.0


In [58]:
print(df.shape)

(13844, 19)


In [60]:
print(df.columns.tolist())
print(df.dtypes)

['journey_date', 'train_number', 'train_name', 'train_type', 'category', 'source_station', 'destination_station', 'station_sequence', 'station_code', 'station_name', 'distance_km', 'platform', 'status', 'scheduled_arrival', 'actual_arrival', 'scheduled_departure', 'actual_departure', 'arrival_delay_minutes', 'departure_delay_minutes']
journey_date                object
train_number                 int64
train_name                  object
train_type                  object
category                    object
source_station              object
destination_station         object
station_sequence             int64
station_code                object
station_name                object
distance_km                float64
platform                   float64
status                      object
scheduled_arrival           object
actual_arrival              object
scheduled_departure         object
actual_departure            object
arrival_delay_minutes      float64
departure_delay_minutes    float6

### Fixing DataTypes

In [61]:
# Fix journey_date
df['journey_date'] = pd.to_datetime(df['journey_date'], dayfirst=True)

# Fix train_number - no decimals needed
df['train_number'] = df['train_number'].astype(int).astype(str)

# Fix timestamp columns - remove timezone info
df['scheduled_arrival'] = pd.to_datetime(df['scheduled_arrival'], utc=True).dt.tz_localize(None)
df['actual_arrival'] = pd.to_datetime(df['actual_arrival'], utc=True).dt.tz_localize(None)
df['scheduled_departure'] = pd.to_datetime(df['scheduled_departure'], utc=True).dt.tz_localize(None)
df['actual_departure'] = pd.to_datetime(df['actual_departure'], utc=True).dt.tz_localize(None)

print(df.dtypes)

journey_date               datetime64[ns]
train_number                       object
train_name                         object
train_type                         object
category                           object
source_station                     object
destination_station                object
station_sequence                    int64
station_code                       object
station_name                       object
distance_km                       float64
platform                          float64
status                             object
scheduled_arrival          datetime64[ns]
actual_arrival             datetime64[ns]
scheduled_departure        datetime64[ns]
actual_departure           datetime64[ns]
arrival_delay_minutes             float64
departure_delay_minutes           float64
dtype: object


In [62]:
print(df[['arrival_delay_minutes', 'departure_delay_minutes']].describe())

       arrival_delay_minutes  departure_delay_minutes
count           12904.000000             10644.000000
mean               40.192808                45.733183
std               110.737681               109.587187
min                -8.000000                 0.000000
25%                 5.000000                12.000000
50%                15.000000                21.000000
75%                32.000000                38.000000
max              1123.000000              1456.000000


In [63]:
print(df.isnull().sum())

journey_date                  0
train_number                  0
train_name                    0
train_type                    0
category                      0
source_station                0
destination_station           0
station_sequence              0
station_code                  0
station_name                  0
distance_km                   0
platform                     44
status                        0
scheduled_arrival           796
actual_arrival              940
scheduled_departure         796
actual_departure           3200
arrival_delay_minutes       940
departure_delay_minutes    3200
dtype: int64


### Missing_Arrival

In [79]:
# Find rows where arrival is missing but station is NOT the first station
last_seq = df.groupby('train_number')['station_sequence'].transform('max')

missing_arrival = df[(df['arrival_delay_minutes'].isnull()) &
                     (df['station_sequence'] != last_seq) &
                     (df['station_sequence'] != 1)]

print(f"Genuinely missing arrivals: {len(missing_arrival)}")

Genuinely missing arrivals: 144


In [80]:
# Find rows where arrival is missing but station is NOT the first station
last_seq = df.groupby('train_number')['station_sequence'].transform('max')

missing_arrival = df[(df['arrival_delay_minutes'].isnull()) &
                     (df['station_sequence'] == last_seq)]

print(f"Genuinely missing arrivals: {len(missing_arrival)}")

Genuinely missing arrivals: 0


### Missing_Departure


In [66]:
# Find rows where departure is missing but station is NOT the last station
last_seq = df.groupby('train_number')['station_sequence'].transform('max')

missing_departure = df[(df['departure_delay_minutes'].isnull()) &
                       (df['station_sequence'] != last_seq) &
                       (df['station_sequence'] != 1)]

print(f"Genuinely missing departures: {len(missing_departure)}")

Genuinely missing departures: 1905


In [68]:
missing_departure = df[(df['departure_delay_minutes'].isnull()) &
                       (df['station_sequence'] == 1)]

print(f"Genuinely missing departures: {len(missing_departure)}")

Genuinely missing departures: 499


In [69]:
last_seq = df.groupby('train_number')['station_sequence'].transform('max')

missing_departure = df[(df['departure_delay_minutes'].isnull()) &
                       (df['station_sequence'] == last_seq)]

print(f"Genuinely missing departures: {len(missing_departure)}")

Genuinely missing departures: 796


### Fixing arrival and departure data

In [81]:
# Step 1 - First station → both = 0
df.loc[df['station_sequence'] == 1, 'arrival_delay_minutes'] = 0
df.loc[df['station_sequence'] == 1, 'departure_delay_minutes'] = 0

# Step 2 - Last station → departure = 0
last_seq = df.groupby(['train_number', 'journey_date'])['station_sequence'].transform('max')
df.loc[df['station_sequence'] == last_seq, 'departure_delay_minutes'] = 0

# Step 3 - Interpolate both in-between missing
df = df.sort_values(['train_number', 'journey_date', 'station_sequence'])

df['arrival_delay_minutes'] = df.groupby(['train_number', 'journey_date'])\
    ['arrival_delay_minutes']\
    .transform(lambda x: x.interpolate(method='linear'))

df['departure_delay_minutes'] = df.groupby(['train_number', 'journey_date'])\
    ['departure_delay_minutes']\
    .transform(lambda x: x.interpolate(method='linear'))

# Step 4 - Verify
print(f"Missing arrival: {df['arrival_delay_minutes'].isnull().sum()}")
print(f"Missing departure: {df['departure_delay_minutes'].isnull().sum()}")
print(f"Total rows: {len(df)}")

Missing arrival: 0
Missing departure: 0
Total rows: 13844


In [82]:
df.to_csv('../data/processed/station_delays_imputed.csv', index=False)
print("✅ Saved!")

✅ Saved!
